# AWP Financial Data Science Pipeline

Multi-agent stock analysis: fetch real market data (yfinance), compute technical indicators,
generate charts, perform portfolio optimization, and produce a comprehensive investment report.

**Autonomy Level A4** — Manager delegates to specialized workers dynamically.

Configure in **Cell 1**, then **Run All** (Ctrl+Shift+Enter).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  AWP FINANCIAL DATA SCIENCE — EDIT THIS CELL, THEN RUN ALL                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── 1. TASK ─────────────────────────────────────────────────────────────────────
TASK = (
    "You are a quantitative finance analyst. Perform a comprehensive stock analysis "
    "on the provided portfolio of FAANG+ tech stocks (6 months of daily OHLCV data). "
    "Your deliverables:\n\n"
    "1. TECHNICAL ANALYSIS — Compute and plot: 20/50-day SMA, Bollinger Bands, RSI(14), "
    "   MACD(12,26,9). Save each as a separate PNG chart with dark theme.\n\n"
    "2. CORRELATION & RISK — Compute daily returns correlation matrix, plot as heatmap. "
    "   Calculate portfolio Sharpe ratio, max drawdown, Value-at-Risk (95% and 99%). "
    "   Save as risk_analysis.png.\n\n"
    "3. PORTFOLIO OPTIMIZATION — Use mean-variance optimization (Markowitz). "
    "   Find the minimum-variance portfolio and the maximum Sharpe ratio portfolio. "
    "   Plot the efficient frontier. Save as efficient_frontier.png.\n\n"
    "4. PERFORMANCE DASHBOARD — Create a single summary dashboard PNG showing: "
    "   normalized price performance (rebased to 100), cumulative returns, "
    "   rolling 30-day volatility, and a volume bar chart. Save as dashboard.png.\n\n"
    "5. REPORT — Write a comprehensive investment report as report.md (1500+ words) "
    "   with sections: Executive Summary, Technical Analysis, Risk Assessment, "
    "   Portfolio Recommendations, and Outlook. Include actual numbers from the data.\n\n"
    "6. DATA EXPORT — Save computed metrics (returns, volatility, Sharpe, VaR, "
    "   optimal weights) as metrics.json.\n\n"
    "Use matplotlib with a dark background style. All charts must have proper labels, "
    "titles, legends, and gridlines. Use the actual data provided — do not fabricate numbers."
)

# ── 2. INPUTS ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META"]

def _generate_synthetic_data(tickers):
    """Generate realistic synthetic stock data using correlated GBM."""
    np.random.seed(42)
    n_days = 126  # ~6 months trading days
    dates = pd.bdate_range(end=pd.Timestamp.today(), periods=n_days)

    base_prices = {"AAPL": 178, "MSFT": 375, "GOOGL": 140, "AMZN": 178, "NVDA": 480, "META": 360}
    annual_vols = {"AAPL": 0.25, "MSFT": 0.22, "GOOGL": 0.28, "AMZN": 0.30, "NVDA": 0.45, "META": 0.32}
    annual_drifts = {"AAPL": 0.12, "MSFT": 0.15, "GOOGL": 0.08, "AMZN": 0.10, "NVDA": 0.35, "META": 0.18}
    base_volumes = {"AAPL": 55e6, "MSFT": 22e6, "GOOGL": 25e6, "AMZN": 45e6, "NVDA": 40e6, "META": 18e6}

    corr_matrix = np.array([
        [1.00, 0.72, 0.65, 0.58, 0.55, 0.60],
        [0.72, 1.00, 0.70, 0.62, 0.60, 0.55],
        [0.65, 0.70, 1.00, 0.68, 0.58, 0.62],
        [0.58, 0.62, 0.68, 1.00, 0.52, 0.65],
        [0.55, 0.60, 0.58, 0.52, 1.00, 0.50],
        [0.60, 0.55, 0.62, 0.65, 0.50, 1.00],
    ])
    daily_vols = np.array([annual_vols[t] / np.sqrt(252) for t in tickers])
    daily_drifts = np.array([annual_drifts[t] / 252 for t in tickers])
    cov = np.outer(daily_vols, daily_vols) * corr_matrix
    L = np.linalg.cholesky(cov)
    z = np.random.randn(n_days, len(tickers))
    correlated_returns = z @ L.T + daily_drifts

    frames = {}
    for i, ticker in enumerate(tickers):
        prices = [base_prices[ticker]]
        for r in correlated_returns[:, i]:
            prices.append(prices[-1] * (1 + r))
        close = np.array(prices[1:])
        daily_range = close * np.random.uniform(0.005, 0.025, n_days)
        high = close + np.random.uniform(0.3, 0.8, n_days) * daily_range
        low = close - np.random.uniform(0.3, 0.8, n_days) * daily_range
        open_ = low + np.random.uniform(0.2, 0.8, n_days) * (high - low)
        volume = (base_volumes[ticker] * np.random.lognormal(0, 0.3, n_days)).astype(int)
        frames[ticker] = pd.DataFrame({
            "Date": dates, "Open": np.round(open_, 2), "High": np.round(high, 2),
            "Low": np.round(low, 2), "Close": np.round(close, 2), "Volume": volume,
        })
    return frames

# Try yfinance first, fall back to synthetic
stock_frames = None
DATA_SOURCE = "synthetic (GBM with correlations)"

try:
    import yfinance as yf
    print("Fetching real market data via yfinance...")
    raw = yf.download(TICKERS, period="6mo", auto_adjust=True, progress=False)
    if raw is not None and len(raw) > 10:
        stock_frames = {}
        for ticker in TICKERS:
            try:
                df = pd.DataFrame({
                    "Date": raw.index,
                    "Open": raw[("Open", ticker)].values,
                    "High": raw[("High", ticker)].values,
                    "Low": raw[("Low", ticker)].values,
                    "Close": raw[("Close", ticker)].values,
                    "Volume": raw[("Volume", ticker)].values,
                }).dropna()
                if len(df) > 10:
                    stock_frames[ticker] = df
            except (KeyError, TypeError):
                pass
        if len(stock_frames) == len(TICKERS):
            DATA_SOURCE = "yfinance (live)"
        else:
            print(f"  Only got {len(stock_frames)}/{len(TICKERS)} tickers, using synthetic data.")
            stock_frames = None
    else:
        print("  Empty response from yfinance, using synthetic data.")
except Exception as e:
    print(f"  yfinance unavailable ({e.__class__.__name__}: {e})")

if stock_frames is None:
    print("Generating synthetic stock data (correlated GBM)...")
    stock_frames = _generate_synthetic_data(TICKERS)

# Combine into a single DataFrame
all_data = []
for ticker, df in stock_frames.items():
    df_copy = df.copy()
    df_copy["Ticker"] = ticker
    all_data.append(df_copy)
combined_df = pd.concat(all_data, ignore_index=True)

portfolio_config = {
    "tickers": TICKERS,
    "risk_free_rate": 0.05,
    "benchmark": "equal_weight",
    "initial_weights": {t: round(1.0 / len(TICKERS), 4) for t in TICKERS},
    "rebalance_frequency": "monthly",
    "data_source": DATA_SOURCE,
}

INPUTS = {
    "stock_data": combined_df,
    "portfolio_config": portfolio_config,
}

print(f"\nData source: {DATA_SOURCE}")
print(f"Tickers:     {TICKERS}")
print(f"Date range:  {combined_df['Date'].min().date()} to {combined_df['Date'].max().date()}")
print(f"Total rows:  {len(combined_df):,}")
for t in TICKERS:
    sub = combined_df[combined_df["Ticker"] == t]
    pct = (sub["Close"].iloc[-1] / sub["Close"].iloc[0] - 1) * 100
    print(f"  {t}: ${sub['Close'].iloc[0]:.2f} -> ${sub['Close'].iloc[-1]:.2f} ({pct:+.1f}%)")

# ── 3. MODEL ────────────────────────────────────────────────────────────────────
import os
MODEL        = "openrouter/openai/gpt-5-nano"
WORKER_MODEL = None

# ── 4. SECRETS ──────────────────────────────────────────────────────────────────
SECRETS = {}

# ── 5-6. SKILLS & EXTERNAL TOOLS ───────────────────────────────────────────────
SKILLS = []
EXTERNAL_TOOLS = []

# ── 7. BUDGET ───────────────────────────────────────────────────────────────────
MAX_LOOPS      = 50
MAX_TOKENS     = 800_000
MAX_WALLTIME   = 1800
MAX_TOOL_CALLS = 150
MAX_WORKERS    = 50
MAX_DEPTH      = 5

# ── 8. SANDBOX & PACKAGES ──────────────────────────────────────────────────────
SANDBOX  = "subprocess"
PACKAGES = ["matplotlib", "scipy"]

# ── 9. WORKER CAPABILITIES ─────────────────────────────────────────────────────
CODE_MODE     = True
TOOL_CREATION = True
VERBOSE       = True

# ── 10. TOOLS ───────────────────────────────────────────────────────────────────
TOOLS = [
    "code.execute", "file.read", "file.write", "file.list", "file.delete",
    "arithmetic.add", "arithmetic.subtract", "arithmetic.multiply", "arithmetic.divide",
]
FORBIDDEN_TOOLS = ["shell.execute"]

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  END OF CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

## Input Data Preview

In [ ]:
# ── Quick preview of the data going into the agents ────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor="#0d1117")
for ax in axes.flat:
    ax.set_facecolor("#0d1117")

# 1. Normalized prices
ax1 = axes[0, 0]
for t in TICKERS:
    sub = combined_df[combined_df["Ticker"] == t].sort_values("Date")
    normalized = sub["Close"] / sub["Close"].iloc[0] * 100
    ax1.plot(sub["Date"], normalized, label=t, linewidth=1.5)
ax1.set_title("Normalized Price (base=100)", color="white", fontsize=12)
ax1.legend(fontsize=8, facecolor="#161b22", edgecolor="#30363d", labelcolor="white")
ax1.tick_params(colors="white")
ax1.grid(alpha=0.15, color="white")

# 2. Volume comparison
ax2 = axes[0, 1]
avg_vols = {t: combined_df[combined_df["Ticker"] == t]["Volume"].mean() / 1e6 for t in TICKERS}
colors = ["#40C4FF", "#E040FB", "#00E676", "#FFD600", "#FF9100", "#FF1744"]
ax2.bar(avg_vols.keys(), avg_vols.values(), color=colors, edgecolor="none")
ax2.set_title("Avg Daily Volume (M shares)", color="white", fontsize=12)
ax2.tick_params(colors="white")
ax2.grid(alpha=0.15, color="white", axis="y")

# 3. Daily returns distribution
ax3 = axes[1, 0]
for i, t in enumerate(TICKERS):
    sub = combined_df[combined_df["Ticker"] == t].sort_values("Date")
    returns = sub["Close"].pct_change().dropna()
    ax3.hist(returns, bins=30, alpha=0.5, label=t, color=colors[i])
ax3.set_title("Daily Returns Distribution", color="white", fontsize=12)
ax3.legend(fontsize=8, facecolor="#161b22", edgecolor="#30363d", labelcolor="white")
ax3.tick_params(colors="white")
ax3.grid(alpha=0.15, color="white")

# 4. Price range (high-low spread)
ax4 = axes[1, 1]
spreads = {}
for t in TICKERS:
    sub = combined_df[combined_df["Ticker"] == t]
    spreads[t] = ((sub["High"] - sub["Low"]) / sub["Close"] * 100).mean()
ax4.barh(list(spreads.keys()), list(spreads.values()), color=colors, edgecolor="none")
ax4.set_title("Avg Daily Range (% of Close)", color="white", fontsize=12)
ax4.tick_params(colors="white")
ax4.grid(alpha=0.15, color="white", axis="x")

plt.tight_layout()
plt.savefig("input_preview.png", dpi=120, facecolor="#0d1117", bbox_inches="tight")
plt.show()
print(f"\nData shape: {combined_df.shape}")
print(combined_df.groupby("Ticker")["Close"].describe().round(2))

## Execute Pipeline

In [ ]:
# ── Setup & Execute ─────────────────────────────────────────
import os, time, shutil, subprocess, sys
from pathlib import Path
from dotenv import load_dotenv

try:
    import awp
except ImportError:
    print("Installing awp-agents from PyPI...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "awp-agents[data]", "--quiet"])
    import awp

for env_path in [Path.home() / "projects" / "awp" / ".env", Path(".env")]:
    if env_path.exists():
        load_dotenv(env_path)
        break

if "OPENROUTER_API_KEY" not in os.environ:
    raise RuntimeError(
        "OPENROUTER_API_KEY not found.\n"
        "Create a .env file with: OPENROUTER_API_KEY=sk-or-v1-...\n"
        "Or set it: os.environ['OPENROUTER_API_KEY'] = 'your-key'"
    )
os.environ["LLM_API_KEY"] = os.environ["OPENROUTER_API_KEY"]

from awp.data import AgentWorkflow

OUTPUT_DIR = (Path.cwd() / "output_finance").resolve()
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_secrets = {k: v for k, v in SECRETS.items() if v} or None

print(f"Model:      {MODEL}")
print(f"Task:       {TASK[:80]}...")
print(f"Data:       {DATA_SOURCE}")
print(f"Inputs:     {list(INPUTS.keys())}")
print(f"Budget:     loops={MAX_LOOPS}, tokens={MAX_TOKENS:,}, wall={MAX_WALLTIME}s")
print(f"Sandbox:    {SANDBOX} + {PACKAGES}")
print(f"Output:     {OUTPUT_DIR}")
print()

t0 = time.time()

result = AgentWorkflow(
    inputs=INPUTS,
    task=TASK,
    model=MODEL,
    worker_model=WORKER_MODEL,
    max_loops=MAX_LOOPS,
    max_total_tokens=MAX_TOKENS,
    max_wall_time=MAX_WALLTIME,
    max_tool_calls=MAX_TOOL_CALLS,
    max_total_workers=MAX_WORKERS,
    max_depth=MAX_DEPTH,
    sandbox=SANDBOX,
    packages=PACKAGES,
    code_mode=CODE_MODE,
    tool_creation=TOOL_CREATION,
    tools=TOOLS,
    forbidden_tools=FORBIDDEN_TOOLS,
    secrets=_secrets,
    skills=SKILLS or None,
    external_tools=EXTERNAL_TOOLS or None,
    output_dir=str(OUTPUT_DIR),
    verbose=VERBOSE,
).run()

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"Done in {elapsed:.1f}s — Status: {result['status']}")
print(f"{'='*60}")

## Results Summary

In [ ]:
# ── Status overview ─────────────────────────────────────────
meta = result["metadata"]
status = result["status"]
status_icon = "\u2705" if status == "complete" else "\u274c"

print(f"{status_icon} Status:      {status}")
print(f"   Run ID:      {meta['run_id']}")
print(f"   Loops:       {meta['loops']}")
print(f"   Wall time:   {meta['wall_time']:.1f}s ({meta['wall_time']/60:.1f} min)")
print(f"   Workers:     {meta['workers_spawned']}")
print(f"   Tool calls:  {meta['tool_calls']}")
print(f"   Tokens:      {meta['tokens_used']:,}")
print(f"   Artifacts:   {len(result['artifacts'])} files")

r = result["result"]
if isinstance(r, dict):
    print(f"\n   Confidence:  {r.get('confidence', 'N/A')}")
    if "termination_reason" in r:
        print(f"   Terminated:  {r['termination_reason']}")
    if "history_summary" in r:
        hist = r["history_summary"]
        confs = [h.get("confidence", "?") for h in hist]
        print(f"   Conf. curve: {' -> '.join(str(c) for c in confs)}")

## Charts & Visualizations

In [ ]:
# ── Display all generated PNG charts ───────────────────────
from IPython.display import display, Image, Markdown

png_files = sorted(
    f for f in OUTPUT_DIR.rglob("*.png")
    if f.stat().st_size > 100
)

if not png_files:
    print("No PNG charts found in output directory.")
else:
    print(f"Found {len(png_files)} chart(s):\n")
    for png in png_files:
        rel = png.relative_to(OUTPUT_DIR)
        size_kb = png.stat().st_size / 1024
        display(Markdown(f"### `{rel}` ({size_kb:.0f} KB)"))
        display(Image(filename=str(png), width=900))
        print()

## Investment Report

In [ ]:
# ── Display markdown reports ───────────────────────────────
from IPython.display import display, Markdown

output_path = OUTPUT_DIR / "output"
md_files = sorted(output_path.rglob("*.md")) if output_path.exists() else []
txt_files = sorted(output_path.rglob("*.txt")) if output_path.exists() else []

if md_files:
    print(f"Found {len(md_files)} report file(s):\n")
    for md_file in md_files:
        rel = md_file.relative_to(OUTPUT_DIR)
        content = md_file.read_text(encoding="utf-8")
        word_count = len(content.split())
        display(Markdown(f"---\n### `{rel}` ({word_count} words)\n\n{content}"))
elif txt_files:
    for txt_file in txt_files:
        rel = txt_file.relative_to(OUTPUT_DIR)
        content = txt_file.read_text(encoding="utf-8")
        print(f"--- {rel} ---")
        print(content)
else:
    print("No report files generated.")

## Metrics & JSON Output

In [ ]:
# ── Display JSON metrics if generated ──────────────────────
import json

output_path = OUTPUT_DIR / "output"
json_files = sorted(output_path.rglob("*.json")) if output_path.exists() else []

if json_files:
    for jf in json_files:
        rel = jf.relative_to(OUTPUT_DIR)
        try:
            data = json.loads(jf.read_text(encoding="utf-8"))
            print(f"━━━ {rel} ━━━")
            print(json.dumps(data, indent=2, default=str)[:3000])
            print()
        except json.JSONDecodeError:
            print(f"  {rel}: (invalid JSON)")
else:
    print("No JSON metric files found.")

## Delegation Loop Logs

In [ ]:
# ── Show delegation loop iteration details ────────────────
import json
from IPython.display import display, Markdown

runs_dir = OUTPUT_DIR / "workspace" / "runs"
if not runs_dir.exists():
    print("No run logs found.")
else:
    run_dir = sorted(runs_dir.iterdir())[-1]
    iterations_dir = run_dir / "iterations"
    
    if not iterations_dir.exists():
        print("No iteration logs found.")
    else:
        iter_dirs = sorted(d for d in iterations_dir.iterdir() if d.is_dir())
        print(f"Total iterations: {len(iter_dirs)}\n")
        
        for iter_d in iter_dirs:
            iter_num = iter_d.name
            
            # Manager decision
            decision_file = iter_d / "manager_decision.json"
            if decision_file.exists():
                dec = json.loads(decision_file.read_text())
                decision = dec.get("decision", "?")
                confidence = dec.get("confidence", "?")
                reasoning = dec.get("reasoning", "")[:200]
                n_workers = len(dec.get("workers", []))
                
                icon = {"delegate": "🔄", "complete": "✅", "fail": "❌"}.get(decision, "❓")
                print(f"{'━'*70}")
                print(f"{icon} Iteration {iter_num}: {decision.upper()} (conf: {confidence})")
                if reasoning:
                    print(f"   Reasoning: {reasoning}")
                if n_workers:
                    print(f"   Workers dispatched: {n_workers}")
            
            # Budget snapshot
            budget_file = iter_d / "budget_snapshot.json"
            if budget_file.exists():
                budget = json.loads(budget_file.read_text())
                remaining = budget.get("budget_remaining_pct", "?")
                print(f"   Budget remaining: {remaining}%")
            
            # Worker results
            deleg_dir = iter_d / "delegations"
            if deleg_dir.exists():
                for worker_dir in sorted(deleg_dir.iterdir()):
                    if not worker_dir.is_dir():
                        continue
                    worker_name = worker_dir.name
                    
                    # Envelope
                    env_file = worker_dir / "envelope.json"
                    if env_file.exists():
                        env = json.loads(env_file.read_text())
                        instructions = env.get("instructions", "")[:150]
                        tools = env.get("tools_allowed", [])
                        print(f"\n   Worker: {worker_name}")
                        print(f"     Instructions: {instructions}...")
                        print(f"     Tools: {', '.join(tools[:5])}")
                    
                    # Result
                    result_file = worker_dir / "result.json"
                    if result_file.exists():
                        res = json.loads(result_file.read_text())
                        w_conf = res.get("confidence", "?")
                        w_err = res.get("error")
                        output_keys = [k for k in res if k not in (
                            "_tool_calls", "tools_created", "confidence", 
                            "error", "_confidence_source"
                        )]
                        status_icon = "❌" if w_err else "✅"
                        print(f"     {status_icon} Confidence: {w_conf}")
                        print(f"     Output keys: {output_keys}")
                        if w_err:
                            print(f"     Error: {str(w_err)[:200]}")
                    
                    # Tool calls
                    tc_file = worker_dir / "tool_calls.json"
                    if tc_file.exists():
                        tc = json.loads(tc_file.read_text())
                        if isinstance(tc, list):
                            ok = sum(1 for t in tc if t.get("result", {}).get("ok"))
                            fail = len(tc) - ok
                            print(f"     Tool calls: {len(tc)} ({ok} ok, {fail} failed)")
            print()

## Run Summary (Markdown Log)

In [ ]:
# ── Display the human-readable RUN_SUMMARY.md ─────────────
from IPython.display import display, Markdown

runs_dir = OUTPUT_DIR / "workspace" / "runs"
if runs_dir.exists():
    run_dir = sorted(runs_dir.iterdir())[-1]
    summary_file = run_dir / "RUN_SUMMARY.md"
    if summary_file.exists():
        content = summary_file.read_text(encoding="utf-8")
        display(Markdown(content))
    else:
        print(f"No RUN_SUMMARY.md in {run_dir.name}")
else:
    print("No run logs found.")

## All Artifacts

In [ ]:
# ── List all generated files with sizes ────────────────────
output_path = OUTPUT_DIR / "output"
if output_path.exists():
    total = 0
    files = sorted(output_path.rglob("*"))
    for f in files:
        if f.is_file():
            sz = f.stat().st_size
            total += sz
            rel = str(f.relative_to(OUTPUT_DIR))
            ext = f.suffix.lower()
            icon = {".png": "🖼️", ".md": "📄", ".json": "📊", ".csv": "📋", ".txt": "📝"}.get(ext, "📎")
            marker = "✅" if sz > 100 else "⚠️"
            print(f"  {marker} {icon} {rel:<55s} {sz:>8,} bytes")
    print(f"\n  Total: {total:,} bytes in {sum(1 for f in files if f.is_file())} files")
else:
    print("No output directory found.")

# Also show workspace files
ws_path = OUTPUT_DIR / "workspace"
if ws_path.exists():
    ws_files = sorted(ws_path.rglob("*"))
    ws_total = sum(f.stat().st_size for f in ws_files if f.is_file())
    ws_count = sum(1 for f in ws_files if f.is_file())
    print(f"\n  Workspace: {ws_total:,} bytes in {ws_count} files")

## Raw Result (JSON)

In [ ]:
# ── Full raw result JSON (truncated for readability) ──────
import json

print(json.dumps(result, indent=2, default=str)[:5000])
if len(json.dumps(result, default=str)) > 5000:
    print(f"\n... (truncated, full result is {len(json.dumps(result, default=str)):,} chars)")